# CUHK-X Small Model Track — Step 2: train

Trains TSM-ResNet18 per modality with **GroupKFold on user**, so validation
measures the thing the competition actually scores: generalisation to people
the model has never seen.

**Setup:** attach the `compact` dataset produced by Step 1. Accelerator:
**GPU P100** (or T4 x2). Internet: On (for the ImageNet weights).

Budget guide on a P100 — roughly 6–10 min/epoch for ~15k clips at 16x144.
One modality x 20 epochs ≈ 2–3 h. You have 30 GPU-h/week, so plan the order:
**Thermal first** (strongest single modality in the source paper: 92.6%),
then Depth_Color, then IR.

In [ ]:
import os, sys, json, time, glob, subprocess
DATA = "/kaggle/input/cuhkx-compact/compact"     # <- adjust to your dataset path
if not os.path.isdir(DATA):
    cands = glob.glob("/kaggle/input/*/compact") + glob.glob("/kaggle/input/*")
    print("compact/ not at the default path. Candidates:")
    for c in cands[:20]: print("  ", c)
    DATA = cands[0] if cands else DATA
print("DATA =", DATA)
print(sorted(os.listdir(DATA))[:10])
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

## Library

In [ ]:
%%writefile cuhkx.py
"""
CUHK-X Small Model Track — shared training/inference library.

Design notes (why it is built this way):

* The competition is scored cross-subject: test users never appear in training.
  So validation MUST be grouped by user, otherwise CV is meaningless and you
  tune yourself off a cliff. Everything here uses GroupKFold on `user`.

* Backbone is ImageNet-pretrained ResNet18 with Temporal Shift Modules inserted
  into the residual branches. TSM gives 3D-conv-like temporal modelling at 2D
  cost and adds *zero* parameters, which matters under the 100 MB budget.
  Host confirmed ImageNet-pretrained small CNNs are allowed.

* Per-clip intensity normalisation is applied to thermal/IR. Absolute pixel
  level encodes body temperature and ambient conditions — i.e. subject and
  session identity — which is exactly the nuisance variable we must discard to
  generalise across people.

* Weights are exported fp16; a full modality x fold ensemble packs into one
  checkpoint well under 100 MB.
"""
from __future__ import annotations

import json
import math
import os
import random
from dataclasses import dataclass, field, asdict

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

cv2.setNumThreads(0)

NUM_CLASSES = 40


# ------------------------------------------------------------------- config

@dataclass
class Cfg:
    data_root: str = "/kaggle/input/cuhkx-compact"
    modality: str = "Thermal"
    frames: int = 16          # frames fed to the model
    size: int = 144           # train crop; stored frames are larger
    arch: str = "resnet18"
    n_folds: int = 5
    fold: int = 0
    epochs: int = 20
    batch_size: int = 16
    lr: float = 3e-4
    backbone_lr_mult: float = 0.3   # pretrained trunk moves slower than the head
    weight_decay: float = 0.05
    label_smoothing: float = 0.1
    mixup_alpha: float = 0.2
    mixup_prob: float = 0.5
    shift_div: int = 8        # TSM: fraction of channels shifted
    dropout: float = 0.3
    ema_decay: float = 0.999
    warmup_frac: float = 0.1
    grad_clip: float = 5.0
    amp: bool = True
    num_workers: int = 2
    seed: int = 42
    per_clip_norm: bool = True
    out_dir: str = "/kaggle/working"
    extra: dict = field(default_factory=dict)

    def to_json(self):
        return json.dumps(asdict(self), indent=2, default=str)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ------------------------------------------------------------------- dataset

class BlobReader:
    """Random access into the packed JPEG blobs written by 04_preprocess.py."""

    def __init__(self, mod_dir):
        self.mod_dir = mod_dir
        self._files = {}

    def read(self, blob_id, offset, length):
        f = self._files.get(blob_id)
        if f is None:
            f = open(os.path.join(self.mod_dir, f"blob_{blob_id:03d}.bin"), "rb")
            self._files[blob_id] = f
        f.seek(offset)
        return f.read(length)

    def __getstate__(self):
        # File handles must not cross the fork into dataloader workers.
        return {"mod_dir": self.mod_dir, "_files": {}}

    def __setstate__(self, s):
        self.__dict__.update(s)
        self._files = {}


def load_index(data_root, modality):
    mod_dir = os.path.join(data_root, modality)
    idx = pd.read_parquet(os.path.join(mod_dir, "index.parquet"))
    return mod_dir, idx


class ClipDataset(Dataset):
    """
    Yields (clip_tensor[T,C,H,W], label).

    Temporal sampling: the preprocessor already stored T_stored uniformly-spaced
    frames. At train time we jitter *within* that grid so the model sees
    different phases of the action across epochs.
    """

    def __init__(self, df, mod_dir, cfg: Cfg, train: bool):
        self.df = df.reset_index(drop=True)
        self.reader = BlobReader(mod_dir)
        self.cfg = cfg
        self.train = train

    def __len__(self):
        return len(self.df)

    # -- frame decode -------------------------------------------------------
    def _decode(self, row, take):
        offs, lens = row["offsets"], row["lengths"]
        out = []
        for i in take:
            buf = self.reader.read(int(row["blob"]), int(offs[i]), int(lens[i]))
            a = np.frombuffer(buf, np.uint8)
            img = cv2.imdecode(a, cv2.IMREAD_COLOR)
            if img is None:
                img = np.zeros((self.cfg.size, self.cfg.size, 3), np.uint8)
            out.append(img)
        return out

    def _pick(self, n_stored):
        T = self.cfg.frames
        if n_stored <= T:
            base = list(range(n_stored)) + [n_stored - 1] * (T - n_stored)
            return base
        if self.train:
            # segment-based random sampling (TSN style): one random frame per segment
            edges = np.linspace(0, n_stored, T + 1)
            return [int(np.random.randint(edges[i], max(edges[i] + 1, edges[i + 1])))
                    for i in range(T)]
        edges = np.linspace(0, n_stored, T + 1)
        return [int((edges[i] + edges[i + 1]) / 2) for i in range(T)]

    # -- augmentation -------------------------------------------------------
    def _augment(self, imgs):
        cfg = self.cfg
        H, W = imgs[0].shape[:2]
        if self.train:
            scale = np.random.uniform(0.65, 1.0)
            ar = np.random.uniform(0.85, 1.18)
            ch = int(min(H, H * scale * ar))
            cw = int(min(W, W * scale / ar))
            y0 = np.random.randint(0, H - ch + 1)
            x0 = np.random.randint(0, W - cw + 1)
            flip = np.random.rand() < 0.5
            # brightness/contrast jitter — models sensor gain drift, not identity
            alpha = np.random.uniform(0.85, 1.15)
            beta = np.random.uniform(-12, 12)
        else:
            side = int(min(H, W) * 0.90)
            y0 = (H - side) // 2
            x0 = (W - side) // 2
            ch = cw = side
            flip, alpha, beta = False, 1.0, 0.0

        out = []
        for im in imgs:
            im = im[y0:y0 + ch, x0:x0 + cw]
            im = cv2.resize(im, (cfg.size, cfg.size), interpolation=cv2.INTER_LINEAR)
            if flip:
                im = im[:, ::-1]
            if self.train and (alpha != 1.0 or beta != 0.0):
                im = np.clip(im.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
            out.append(im)
        return out

    def __getitem__(self, i):
        row = self.df.iloc[i]
        take = self._pick(int(row["t"]))
        imgs = self._decode(row, take)
        imgs = self._augment(imgs)

        x = np.stack(imgs).astype(np.float32) / 255.0     # [T,H,W,3]
        x = torch.from_numpy(x).permute(0, 3, 1, 2)        # [T,3,H,W]

        if self.cfg.per_clip_norm:
            # Standardise each clip independently: removes the absolute thermal /
            # IR offset that identifies the subject and the session.
            m, s = x.mean(), x.std().clamp_min(1e-4)
            x = (x - m) / s
        else:
            mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
            x = (x - mean) / std

        if self.train and np.random.rand() < 0.25:
            x = random_erase(x)

        return x, int(row["action_id"])


def random_erase(x, max_frac=0.20):
    T, C, H, W = x.shape
    h = int(H * np.random.uniform(0.06, max_frac))
    w = int(W * np.random.uniform(0.06, max_frac))
    y = np.random.randint(0, H - h + 1)
    xx = np.random.randint(0, W - w + 1)
    x[:, :, y:y + h, xx:xx + w] = torch.randn(T, C, h, w) * 0.1
    return x


# --------------------------------------------------------------------- model

class TemporalShift(nn.Module):
    """
    Shift a fraction of channels forward/backward along time before `block`.

    Cost: a memory copy. Params added: zero. This is what buys temporal
    reasoning without paying for 3D convolutions.
    """

    def __init__(self, block, n_segment, shift_div=8):
        super().__init__()
        self.block = block
        self.n_segment = n_segment
        self.shift_div = shift_div

    def forward(self, x):
        nt, c, h, w = x.size()
        t = self.n_segment
        n = nt // t
        x = x.view(n, t, c, h, w)
        fold = max(1, c // self.shift_div)
        out = torch.zeros_like(x)
        out[:, :-1, :fold] = x[:, 1:, :fold]              # shift left  (future)
        out[:, 1:, fold:2 * fold] = x[:, :-1, fold:2 * fold]  # shift right (past)
        out[:, :, 2 * fold:] = x[:, :, 2 * fold:]         # keep the rest
        return self.block(out.view(nt, c, h, w))


def make_tsm_resnet(arch="resnet18", n_segment=16, shift_div=8, pretrained=True):
    import torchvision
    fn = getattr(torchvision.models, arch)
    try:
        net = fn(weights="IMAGENET1K_V1" if pretrained else None)
    except TypeError:
        net = fn(pretrained=pretrained)

    # Wrap the first conv of every BasicBlock so the shift happens on the
    # residual branch only — identity path stays clean (as in the TSM paper).
    for layer in [net.layer1, net.layer2, net.layer3, net.layer4]:
        for blk in layer:
            blk.conv1 = TemporalShift(blk.conv1, n_segment, shift_div)
    feat_dim = net.fc.in_features
    net.fc = nn.Identity()
    return net, feat_dim


class VideoNet(nn.Module):
    """TSM backbone + attention-weighted temporal pooling + linear classifier."""

    def __init__(self, cfg: Cfg, num_classes=NUM_CLASSES, pretrained=True):
        super().__init__()
        self.cfg = cfg
        self.backbone, d = make_tsm_resnet(
            cfg.arch, cfg.frames, cfg.shift_div, pretrained)
        self.attn = nn.Sequential(nn.Linear(d, d // 4), nn.Tanh(), nn.Linear(d // 4, 1))
        self.drop = nn.Dropout(cfg.dropout)
        self.fc = nn.Linear(d, num_classes)
        nn.init.trunc_normal_(self.fc.weight, std=0.01)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):                 # x: [B,T,3,H,W]
        B, T = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))         # [B*T, d]
        f = f.view(B, T, -1)
        a = self.attn(f).softmax(dim=1)            # [B,T,1]
        pooled = (f * a).sum(1)                    # [B,d]
        return self.fc(self.drop(pooled))


# ------------------------------------------------------- skeleton (ST-GCN)

# Human3.6M 17-joint bone list; parent[j] is the joint j hangs off.
H36M_EDGES = [(0, 1), (1, 2), (2, 3), (0, 4), (4, 5), (5, 6),
              (0, 7), (7, 8), (8, 9), (9, 10),
              (8, 11), (11, 12), (12, 13), (8, 14), (14, 15), (15, 16)]
N_JOINTS = 17
PARENT = np.zeros(N_JOINTS, dtype=np.int64)
for _p, _c in H36M_EDGES:
    PARENT[_c] = _p


def build_adjacency():
    """
    Three-partition spatial graph (ST-GCN): self, centripetal (towards root),
    centrifugal (away). Splitting by distance-to-root lets the convolution treat
    "limb moving inward" and "limb moving outward" differently, which matters
    for reach/retract actions like *Take medicine* vs *Put on clothes*.
    """
    A = np.zeros((N_JOINTS, N_JOINTS), np.float32)
    for i, j in H36M_EDGES:
        A[i, j] = A[j, i] = 1.0

    # hop distance from the root joint
    dist = np.full(N_JOINTS, 1e9)
    dist[0] = 0
    for _ in range(N_JOINTS):
        for i, j in H36M_EDGES:
            dist[j] = min(dist[j], dist[i] + 1)
            dist[i] = min(dist[i], dist[j] + 1)

    parts = np.zeros((3, N_JOINTS, N_JOINTS), np.float32)
    parts[0] = np.eye(N_JOINTS, dtype=np.float32)
    for i in range(N_JOINTS):
        for j in range(N_JOINTS):
            if A[i, j] == 0:
                continue
            if dist[j] < dist[i]:
                parts[1, i, j] = 1.0      # neighbour closer to root
            else:
                parts[2, i, j] = 1.0      # neighbour further from root

    # symmetric normalisation, per partition
    for k in range(3):
        d = parts[k].sum(1, keepdims=True)
        parts[k] = parts[k] / np.maximum(d, 1e-6)
    return torch.from_numpy(parts)


def pose_features(x):
    """
    [B,T,V,3] -> [B,9,T,V]: joint position, bone vector, and velocity.

    Bones encode limb orientation independently of where the joint sits, and
    velocity supplies the short-term dynamics that separate the exercise classes
    (jog / squat / jumping jack) from each other.
    """
    B, T, V, C = x.shape
    parent = torch.as_tensor(PARENT, device=x.device)
    bone = x - x[:, :, parent, :]
    vel = torch.zeros_like(x)
    vel[:, 1:] = x[:, 1:] - x[:, :-1]
    f = torch.cat([x, bone, vel], dim=-1)          # [B,T,V,9]
    return f.permute(0, 3, 1, 2).contiguous()      # [B,9,T,V]


class STGCNBlock(nn.Module):
    def __init__(self, cin, cout, A, stride=1, dropout=0.1, residual=True):
        super().__init__()
        self.register_buffer("A", A)
        K = A.size(0)
        self.gcn = nn.Conv2d(cin, cout * K, 1)
        self.K, self.cout = K, cout
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(cout), nn.Dropout(dropout),
        )
        # learnable edge weighting — lets the graph adapt beyond the skeleton
        self.edge = nn.Parameter(torch.ones(K, A.size(1), A.size(2)))
        if not residual:
            self.res = None
        elif cin == cout and stride == 1:
            self.res = nn.Identity()
        else:
            self.res = nn.Sequential(nn.Conv2d(cin, cout, 1, (stride, 1)),
                                     nn.BatchNorm2d(cout))
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        res = 0 if self.res is None else self.res(x)
        y = self.gcn(x)
        N, _, T, V = y.shape
        y = y.view(N, self.K, self.cout, T, V)
        y = torch.einsum("nkctv,kvw->nctw", y, self.A * self.edge)
        return self.relu(self.tcn(y) + res)


class SkeletonNet(nn.Module):
    """
    Compact ST-GCN over 3D pose. ~1 M parameters, so it costs almost nothing
    against the 100 MB budget while adding a modality that is present for 100%
    of clips and is far more subject-invariant than appearance.
    """

    def __init__(self, num_classes=NUM_CLASSES, width=64, dropout=0.3):
        super().__init__()
        A = build_adjacency()
        w = width
        self.bn = nn.BatchNorm1d(9 * N_JOINTS)
        self.blocks = nn.ModuleList([
            STGCNBlock(9, w, A, residual=False),
            STGCNBlock(w, w, A),
            STGCNBlock(w, w, A),
            STGCNBlock(w, 2 * w, A, stride=2),
            STGCNBlock(2 * w, 2 * w, A),
            STGCNBlock(2 * w, 4 * w, A, stride=2),
            STGCNBlock(4 * w, 4 * w, A),
        ])
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(4 * w, num_classes)

    def forward(self, x):                       # x: [B,T,V,3]
        f = pose_features(x)                    # [B,9,T,V]
        B, C, T, V = f.shape
        f = self.bn(f.permute(0, 1, 3, 2).reshape(B, C * V, T))
        f = f.view(B, C, V, T).permute(0, 1, 3, 2).contiguous()
        for b in self.blocks:
            f = b(f)
        f = f.mean(dim=(2, 3))                  # global average over time+joints
        return self.fc(self.drop(f))


class SkeletonDataset(Dataset):
    """Yields ([T,17,3] pose, label) from the packed poses.npy."""

    def __init__(self, df, mod_dir, cfg: Cfg, train: bool):
        self.df = df.reset_index(drop=True)
        self.poses = np.load(os.path.join(mod_dir, "poses.npy"), mmap_mode="r")
        self.cfg = cfg
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        seq = np.asarray(self.poses[int(row["row"])], dtype=np.float32)  # [T,V,3]
        T = self.cfg.frames

        if seq.shape[0] != T:                       # resample along time
            src = np.linspace(0, seq.shape[0] - 1, T)
            if self.train:
                src = np.clip(src + np.random.uniform(-0.5, 0.5, T), 0,
                              seq.shape[0] - 1)
            seq = seq[np.round(src).astype(int)]

        if self.train:
            # Rotation about the vertical axis: the camera yaw relative to the
            # subject is arbitrary, so the label must be invariant to it.
            th = np.random.uniform(-0.35, 0.35)
            c, s = np.cos(th), np.sin(th)
            R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], np.float32)
            seq = seq @ R.T
            seq = seq * np.random.uniform(0.9, 1.1)
            seq = seq + np.random.normal(0, 0.01, seq.shape).astype(np.float32)
            if np.random.rand() < 0.5:              # mirror left/right
                seq[..., 0] *= -1
                swap = np.arange(N_JOINTS)
                for a, b in [(1, 4), (2, 5), (3, 6), (11, 14), (12, 15), (13, 16)]:
                    swap[a], swap[b] = b, a
                seq = seq[:, swap]

        return torch.from_numpy(np.ascontiguousarray(seq)), int(row["action_id"])


def make_model(cfg: Cfg, pretrained=True):
    """Dispatch on modality: pose gets the graph net, images get TSM."""
    if cfg.modality.lower().startswith("skel"):
        return SkeletonNet(dropout=cfg.dropout)
    return VideoNet(cfg, pretrained=pretrained)


def make_dataset(df, mod_dir, cfg: Cfg, train: bool):
    if cfg.modality.lower().startswith("skel"):
        return SkeletonDataset(df, mod_dir, cfg, train)
    return ClipDataset(df, mod_dir, cfg, train)


# ------------------------------------------------------------------ training

class EMA:
    """
    Exponential moving average of weights, with a warmup ramp on the decay.

    The shadow starts as a copy of the *untrained* weights, so a flat 0.999
    decay leaves it pinned near initialisation for the first ~1000 steps — on a
    short run you would evaluate and checkpoint an essentially untrained model.
    Ramping the decay as (1+t)/(10+t) makes the average track closely at first
    and tighten as training proceeds.
    """

    def __init__(self, model, decay):
        self.decay = decay
        self.step = 0
        self.shadow = {k: v.detach().clone().float()
                       for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}

    @torch.no_grad()
    def update(self, model):
        self.step += 1
        d = min(self.decay, (1 + self.step) / (10 + self.step))
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(d).add_(v.detach().float(), alpha=1 - d)

    def copy_to(self, model):
        sd = model.state_dict()
        for k, v in self.shadow.items():
            sd[k].copy_(v.to(sd[k].dtype))


def mixup(x, y, alpha):
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[perm], y, y[perm], lam


def cosine_schedule(step, total, warmup):
    if step < warmup:
        return step / max(warmup, 1)
    p = (step - warmup) / max(total - warmup, 1)
    return 0.5 * (1 + math.cos(math.pi * p))


def build_folds(df, n_folds, seed=42):
    """
    GroupKFold on user. Deterministic, and balanced by user count rather than
    row count so each fold holds out a comparable number of *people*.
    """
    users = sorted(df["user"].astype(str).unique())
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(users))
    assign = {users[u]: i % n_folds for i, u in enumerate(order)}
    return df["user"].astype(str).map(assign).values


@torch.no_grad()
def predict_logits(model, loader, device, amp=True):
    model.eval()
    outs, ys = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        with torch.autocast("cuda", enabled=amp and device.type == "cuda"):
            outs.append(model(x).float().cpu())
        ys.append(y)
    return torch.cat(outs), torch.cat(ys)


def pack_fp16(state_dicts, path, meta=None):
    """Bundle every ensemble member into ONE fp16 checkpoint (rule requirement)."""
    packed = {
        name: {k: v.half() if v.dtype.is_floating_point else v
               for k, v in sd.items()}
        for name, sd in state_dicts.items()
    }
    torch.save({"models": packed, "meta": meta or {}}, path)
    mb = os.path.getsize(path) / 1e6
    print(f"checkpoint: {path}  {mb:.1f} MB  ({len(packed)} models)")
    if mb > 100:
        print("!! OVER the 100 MB limit — drop members or shrink the backbone")
    return mb

In [ ]:
%%writefile train.py
"""
Train one modality x one fold. Run once per (modality, fold) on Kaggle GPU.

    python train.py --modality Thermal --fold 0 --epochs 20

Writes to out_dir:
    {modality}_f{fold}.pt          best EMA weights (fp32, packed later)
    {modality}_f{fold}_oof.npz     OOF logits + labels + clip_ids
    {modality}_f{fold}_test.npz    test logits (TTA-averaged)

The OOF files are what fusion weights are fitted on — never fit them on the
leaderboard, that is how teams end up with a 0.97 public score and a Selection
Stage failure.
"""
import argparse
import os
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from cuhkx import (Cfg, EMA, build_folds, cosine_schedule, load_index,
                   make_dataset, make_model, mixup, predict_logits,
                   seed_everything)


def evaluate(model, loader, device, amp):
    logits, y = predict_logits(model, loader, device, amp)
    acc = (logits.argmax(1) == y).float().mean().item()
    return acc, logits, y


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", default="/kaggle/input/cuhkx-compact")
    ap.add_argument("--modality", default="Thermal")
    ap.add_argument("--fold", type=int, default=0)
    ap.add_argument("--n-folds", type=int, default=5)
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--frames", type=int, default=16)
    ap.add_argument("--size", type=int, default=144)
    ap.add_argument("--arch", default="resnet18")
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--out-dir", default="/kaggle/working")
    ap.add_argument("--no-tta", action="store_true")
    args = ap.parse_args()

    cfg = Cfg(data_root=args.data_root, modality=args.modality, fold=args.fold,
              n_folds=args.n_folds, epochs=args.epochs, batch_size=args.batch_size,
              frames=args.frames, size=args.size, arch=args.arch, lr=args.lr,
              seed=args.seed, num_workers=args.workers, out_dir=args.out_dir)
    seed_everything(cfg.seed + cfg.fold)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(cfg.out_dir, exist_ok=True)
    print(cfg.to_json())

    mod_dir, idx = load_index(cfg.data_root, cfg.modality)
    train_all = idx[idx.split == "train"].copy()
    test_df = idx[idx.split == "test"].copy()
    train_all["fold"] = build_folds(train_all, cfg.n_folds, cfg.seed)

    tr = train_all[train_all.fold != cfg.fold]
    va = train_all[train_all.fold == cfg.fold]
    print(f"\n{cfg.modality} fold {cfg.fold}: "
          f"train {len(tr)} clips / {tr.user.nunique()} users | "
          f"val {len(va)} clips / {va.user.nunique()} users")
    print(f"val users: {sorted(va.user.astype(str).unique())}")
    print(f"test clips: {len(test_df)}\n")

    dl = lambda ds, sh: DataLoader(
        ds, batch_size=cfg.batch_size, shuffle=sh, num_workers=cfg.num_workers,
        pin_memory=True, drop_last=sh, persistent_workers=cfg.num_workers > 0)

    train_ds = make_dataset(tr, mod_dir, cfg, train=True)
    val_ds = make_dataset(va, mod_dir, cfg, train=False)
    test_ds = make_dataset(test_df, mod_dir, cfg, train=False)
    train_dl, val_dl, test_dl = dl(train_ds, True), dl(val_ds, False), dl(test_ds, False)

    model = make_model(cfg).to(device)
    n_par = sum(p.numel() for p in model.parameters())
    print(f"params: {n_par/1e6:.2f} M  ({n_par*2/1e6:.1f} MB fp16)\n")

    head = [p for n, p in model.named_parameters()
            if n.startswith(("fc.", "attn."))]
    trunk = [p for n, p in model.named_parameters()
             if not n.startswith(("fc.", "attn."))]
    # Only an ImageNet-pretrained trunk needs a reduced LR; the graph net is
    # trained from scratch, so slowing it down would just under-fit it.
    trunk_mult = cfg.backbone_lr_mult if not cfg.modality.lower().startswith("skel") else 1.0
    opt = torch.optim.AdamW(
        [{"params": trunk, "lr": cfg.lr * trunk_mult},
         {"params": head, "lr": cfg.lr}],
        weight_decay=cfg.weight_decay)

    steps = max(1, len(train_dl)) * cfg.epochs
    warm = int(steps * cfg.warmup_frac)
    base_lrs = [g["lr"] for g in opt.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp and device.type == "cuda")
    crit = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    ema = EMA(model, cfg.ema_decay)

    best, best_sd, step = -1.0, None, 0
    for ep in range(cfg.epochs):
        model.train()
        t0, tot, seen = time.time(), 0.0, 0
        for x, y in train_dl:
            for g, b in zip(opt.param_groups, base_lrs):
                g["lr"] = b * cosine_schedule(step, steps, warm)
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

            use_mix = np.random.rand() < cfg.mixup_prob
            if use_mix:
                x, ya, yb, lam = mixup(x, y, cfg.mixup_alpha)

            with torch.autocast("cuda", enabled=cfg.amp and device.type == "cuda"):
                out = model(x)
                loss = (lam * crit(out, ya) + (1 - lam) * crit(out, yb)
                        if use_mix else crit(out, y))

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt); scaler.update()
            ema.update(model)

            tot += loss.item() * y.size(0); seen += y.size(0); step += 1

        eval_model = make_model(cfg, pretrained=False).to(device)
        eval_model.load_state_dict(model.state_dict())
        ema.copy_to(eval_model)
        acc, _, _ = evaluate(eval_model, val_dl, device, cfg.amp)

        flag = ""
        if acc > best:
            best = acc
            best_sd = {k: v.detach().cpu().clone()
                       for k, v in eval_model.state_dict().items()}
            flag = "  <-- best"
        print(f"ep {ep+1:2d}/{cfg.epochs}  loss {tot/max(seen,1):.4f}  "
              f"val_acc {acc:.4f}  best {best:.4f}  "
              f"{time.time()-t0:.0f}s{flag}", flush=True)
        del eval_model

    tag = f"{cfg.modality}_f{cfg.fold}"
    torch.save(best_sd, os.path.join(cfg.out_dir, f"{tag}.pt"))

    # Reload the best weights, then dump OOF + test logits for fusion.
    model = make_model(cfg, pretrained=False).to(device)
    model.load_state_dict(best_sd)

    _, oof_logits, oof_y = evaluate(model, val_dl, device, cfg.amp)
    np.savez(os.path.join(cfg.out_dir, f"{tag}_oof.npz"),
             logits=oof_logits.numpy(), y=oof_y.numpy(),
             clip_id=va.clip_id.values, user=va.user.astype(str).values)

    n_views = 1 if args.no_tta else 3
    test_logits = 0
    for v in range(n_views):
        torch.manual_seed(1000 + v)
        np.random.seed(1000 + v)
        # View 0 is the deterministic centre view; extra views re-sample time
        # and jitter the crop. The dataset must be rebuilt per view — mutating
        # a flag on it would never reach forked dataloader workers.
        ds_v = make_dataset(test_df, mod_dir, cfg, train=v > 0)
        dl_v = DataLoader(ds_v, batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True)
        lg, _ = predict_logits(model, dl_v, device, cfg.amp)
        test_logits = test_logits + lg.softmax(1)
    test_logits = (test_logits / n_views).numpy()

    np.savez(os.path.join(cfg.out_dir, f"{tag}_test.npz"),
             probs=test_logits, clip_id=test_df.clip_id.values)

    print(f"\nDONE {tag}: best cross-subject val acc = {best:.4f}")
    with open(os.path.join(cfg.out_dir, "results.txt"), "a") as f:
        f.write(f"{tag}\t{best:.4f}\t{n_par/1e6:.2f}M\n")


if __name__ == "__main__":
    main()

In [ ]:
import importlib, cuhkx
importlib.reload(cuhkx)
import pandas as pd, numpy as np, torch
from cuhkx import load_index, build_folds

# Sanity-check the split before spending GPU hours on it.
mod0 = json.load(open(f"{DATA}/prep_config.json"))["modalities"][0] \
       if os.path.exists(f"{DATA}/prep_config.json") else "Thermal"
_, idx = load_index(DATA, mod0)
tr = idx[idx.split == "train"]
print(f"{mod0}: {len(idx)} clips  ({len(tr)} train, {(idx.split=='test').sum()} test)")
print(f"train users ({tr.user.nunique()}): {sorted(tr.user.astype(str).unique())}")
print(f"\nclass balance: min={tr.action_id.value_counts().min()} "
      f"max={tr.action_id.value_counts().max()} over "
      f"{tr.action_id.nunique()} classes")

f = build_folds(tr, 5, 42)
print("\nfold -> held-out users (these must be disjoint from training users):")
for k in range(5):
    u = sorted(tr[f == k].user.astype(str).unique())
    print(f"  fold {k}: {len(tr[f==k]):>5} clips  users {u}")

## Train

Run one cell per (modality, fold). Each writes `{mod}_f{fold}.pt`,
`_oof.npz` and `_test.npz` into `/kaggle/working`.

Start with a single fold to measure the real cross-subject accuracy and the
per-epoch time before committing GPU quota to the rest.

### Skeleton first — minutes, not hours

The 3D-pose ST-GCN is ~2 M params and trains in a few minutes on ~1 MB of
data. Run it first: it costs almost no quota, is present for 100% of clips,
and gives you a real cross-subject number to calibrate everything else
against. 3D pose also discards appearance entirely, so it is the single most
subject-invariant signal available.

In [ ]:
!python train.py --data-root "$DATA" --modality Skeleton --fold 0 \
    --epochs 40 --batch-size 32 --frames 32 --lr 1e-3 \
    --workers 2 --out-dir /kaggle/working

### Then the image modalities

Depth_Color first (100% coverage, 640x480), then Thermal (strongest single
modality in the source paper at 92.6%, but missing on 10 test clips), then IR.

In [ ]:
MODALITY = "Depth_Color"
FOLD     = 0
EPOCHS   = 20

!python train.py --data-root "$DATA" --modality $MODALITY --fold $FOLD \
    --epochs $EPOCHS --batch-size 16 --frames 16 --size 144 \
    --arch resnet18 --lr 3e-4 --workers 2 --out-dir /kaggle/working

### Remaining runs

Uncomment as quota allows. Two folds per modality is usually enough — the
ensemble gain past that is small relative to the budget it consumes.

In [ ]:
# !python train.py --data-root "$DATA" --modality Thermal     --fold 0 --epochs 20 --out-dir /kaggle/working
# !python train.py --data-root "$DATA" --modality IR          --fold 0 --epochs 20 --out-dir /kaggle/working
# !python train.py --data-root "$DATA" --modality Skeleton    --fold 1 --epochs 40 --frames 32 --batch-size 32 --lr 1e-3 --out-dir /kaggle/working
# !python train.py --data-root "$DATA" --modality Depth_Color --fold 1 --epochs 20 --out-dir /kaggle/working
# !python train.py --data-root "$DATA" --modality Thermal     --fold 1 --epochs 20 --out-dir /kaggle/working

In [ ]:
if os.path.exists("/kaggle/working/results.txt"):
    print(open("/kaggle/working/results.txt").read())
for f in sorted(glob.glob("/kaggle/working/*.pt")):
    print(f"{os.path.basename(f):<28} {os.path.getsize(f)/1e6:7.1f} MB")

## Read the result honestly

The validation number here is **cross-subject** and is the closest thing you
have to the private leaderboard *and* to the Selection Stage re-test. If val
says 0.78 and the public LB says 0.90, trust the 0.78 — the Zoom verification
disqualifies teams whose accuracy drops more than 10 points on fresh subjects.

Next: Step 3 fuses the per-modality probabilities and writes the submission.